<a href="https://colab.research.google.com/github/lavanya-001/CSA6102-DIGITAL-FORENSICS-LAB/blob/main/DFIR_EXPERIMENTS_39.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from datetime import datetime
from collections import Counter

LOG_FMT = "%Y-%m-%d %H:%M:%S"

# Function to build baseline IP for each user
def build_baseline_ips(logs):
    by_user = {}

    for entry in logs:
        by_user.setdefault(entry["user"], []).append(entry["ip"])

    return {
        user: Counter(ips).most_common(1)[0][0]
        for user, ips in by_user.items()
    }

# Function to detect anomalous downloads
def flag_anomalous_downloads(logs, business_start=8, business_end=20):
    baseline = build_baseline_ips(logs)
    flagged = []

    for entry in logs:
        if entry["action"] != "download":
            continue

        ts = datetime.strptime(entry["timestamp"], LOG_FMT)
        reasons = []

        if entry["ip"] != baseline.get(entry["user"]):
            reasons.append("IP differs from user baseline")

        if not (business_start <= ts.hour < business_end):
            reasons.append("Outside business hours")

        if reasons:
            flagged.append({
                **entry,
                "reasons": reasons
            })

    return flagged

# Sample log data
logs = [
    {
        "user": "Alice",
        "ip": "192.168.1.10",
        "action": "download",
        "timestamp": "2026-08-06 10:30:00"
    },
    {
        "user": "Alice",
        "ip": "192.168.1.10",
        "action": "login",
        "timestamp": "2026-08-06 09:00:00"
    },
    {
        "user": "Alice",
        "ip": "10.0.0.5",
        "action": "download",
        "timestamp": "2026-08-06 22:15:00"
    },
    {
        "user": "Bob",
        "ip": "172.16.1.20",
        "action": "download",
        "timestamp": "2026-08-06 11:45:00"
    },
    {
        "user": "Bob",
        "ip": "8.8.8.8",
        "action": "download",
        "timestamp": "2026-08-06 23:10:00"
    }
]

# Run detection
flagged = flag_anomalous_downloads(logs)

# Print results
print("Anomalous Downloads:\n")

for item in flagged:
    print("User      :", item["user"])
    print("IP        :", item["ip"])
    print("Action    :", item["action"])
    print("Timestamp :", item["timestamp"])
    print("Reasons   :", ", ".join(item["reasons"]))
    print("-" * 40)

Anomalous Downloads:

User      : Alice
IP        : 10.0.0.5
Action    : download
Timestamp : 2026-08-06 22:15:00
Reasons   : IP differs from user baseline, Outside business hours
----------------------------------------
User      : Bob
IP        : 8.8.8.8
Action    : download
Timestamp : 2026-08-06 23:10:00
Reasons   : IP differs from user baseline, Outside business hours
----------------------------------------
